In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import qmc


# *Set up configuration for an FC pack line. Arbitrary numbers are provided.* #

In [ ]:
CONFIG = {
    "start_date": "2026-04-01",
    "days": 90,
    "line_id": "PACK-LINE-01",
    "total_stations": 18,
    "seed": 258,
    "base_units_per_packer": 85,
    "backlog_moderate": 400,
    "backlog_high": 700,
    "backlog_severe": 1000
}

baseline_hourly_work = 900    # Baseline hourly work volume for a line
max_stations_down = 5
max_missing_packers = 4       # Absenteeism or reassignments
max_extra_packers = 6         # Max available from other lines
max_diverted_units = 350      # Max work that can realistically be diverted

# *Define the shift based on the hour of the day* #

In [ ]:
def get_shift_info(hour):
    if 6 <= hour < 14:              # Heaviest times are between 6am and 2pm
        return "day", 1.15, 18
    elif 14 <= hour < 22:           # There is an occassional spike between 2pm and 10pm
        return "swing", 1.05, 16
    else:
        return "night", 0.75, 10    # Lightest times are between 10pm and 6am

# *Specify weekday and peak factors that reflect the busiest times of demand* #

In [ ]:
def get_demand_factors(timestamp):
    weekday = timestamp.day_name()
    hour = timestamp.hour

    if weekday in ["Thursday", "Friday"]:   # Demand is highest on Thursdays and Fridays
        weekday_factor = 1.15
    elif weekday in ["Monday", "Tuesday"]:  # Demand is lowest on Mondays and Tuesdays
        weekday_factor = 0.80
    else:
        weekday_factor = 1.00

    if hour in [10, 11, 15, 16, 17]:        # Certain hours during the day have heavier volumes
        peak_factor = 1.25
    else:
        peak_factor = 1.00

    return weekday, weekday_factor, peak_factor

# *Use Latin Hypercube to generate samples that cover wide ranges of worker capacity, demand, and workstation downtime, as well as other variables* #

In [ ]:
def create_lhs_samples(n_rows, seed=222):
    sampler = qmc.LatinHypercube(d=6, seed=seed)
    sample = sampler.random(n=n_rows)

    return pd.DataFrame(
        sample,
        columns=[
            "sample_volume",
            "sample_downtime",
            "sample_attendance",
            "sample_quality",
            "sample_extra_labor",
            "sample_other_line_capacity"
        ]
    )

# *Generate values that represent operational conditions like work planned, machines down, packing rate, and worker attendance* #

In [ ]:
def generate_operational_inputs(row, timestamp, config):
    hour = timestamp.hour

    shift, shift_volume_factor, packers_scheduled = get_shift_info(hour)
    weekday, weekday_factor, peak_factor = get_demand_factors(timestamp)

    planned_work = int(
        (baseline_hourly_work * (row["sample_volume"] + 1))
        * shift_volume_factor
        * weekday_factor
        * peak_factor
        * np.random.normal(1.0, 0.06)
    )

    stations_down = int((row["sample_downtime"] ** 2) * max_stations_down)

    attendance_loss = int(row["sample_attendance"] * max_missing_packers)
    packers_available = max(0, packers_scheduled - attendance_loss)

    avg_units_per_packer = int(config["base_units_per_packer"] * np.random.normal(1.0, 0.05))

    building_extra_packers = int(row["sample_extra_labor"] * max_extra_packers)
    other_line_capacity = int(row["sample_other_line_capacity"] * max_diverted_units)

    return {
        "shift": shift,
        "weekday": weekday,
        "planned_work": planned_work,
        "stations_down": stations_down,
        "packers_scheduled": packers_scheduled,
        "packers_available": packers_available,
        "avg_units_per_packer": avg_units_per_packer,
        "building_extra_packers": building_extra_packers,
        "other_line_capacity": other_line_capacity
    }

# *Balance the packer line depending on backlog, work, available stations, available packers, work that can be diverted, etc.* #

In [ ]:
def apply_line_balancing(
    previous_backlog,
    planned_work,
    packers_available,
    active_stations,
    building_extra_packers,
    other_line_capacity,
    config
):
    added_packers = 0
    work_diverted_out = 0
    scaled_back_work = 0
    intervention_type = "none"

    # How many more packers can this line physically use?
    open_station_slots = max(0, active_stations - packers_available)

    # 1. First response: add packers if backlog is elevated
    if previous_backlog > config["backlog_moderate"]:
        added_packers = min(
            open_station_slots,
            building_extra_packers
        )

        packers_available += added_packers

        if added_packers > 0:
            intervention_type = "added_packers"

    # 2. Second response: divert work if backlog remains high
    if previous_backlog > config["backlog_high"]:
        max_divertible_work = int(planned_work * 0.20)

        work_diverted_out = min(
            max_divertible_work,
            other_line_capacity
        )

        if work_diverted_out > 0:
            if intervention_type == "none":
                intervention_type = "work_diverted"
            else:
                intervention_type += "_and_work_diverted"

    # 3. Last resort: scale back only if labor or stations are unavailable
    no_labor_available = packers_available == 0 and building_extra_packers == 0
    no_station_available = active_stations == 0

    if no_labor_available or no_station_available:
        scaled_back_work = int(planned_work * 0.50)
        intervention_type = "planned_work_scaled_back"

    net_incoming_volume = max(
        0,
        planned_work - work_diverted_out - scaled_back_work
    )

    return {
        "added_packers": added_packers,
        "packers_available_after_support": packers_available,
        "work_diverted_out": work_diverted_out,
        "scaled_back_work": scaled_back_work,
        "net_incoming_volume": net_incoming_volume,
        "intervention_type": intervention_type
    }

# *Depending on throughput, determine if there will be a bottleneck* #

In [ ]:
import math

def calculate_line_throughput(inputs, balancing, previous_backlog, active_stations, config):
    available_work = previous_backlog + balancing["net_incoming_volume"]

    avg_units_per_packer = inputs["avg_units_per_packer"]

    needed_packers = (
        math.ceil(available_work / avg_units_per_packer)
        if avg_units_per_packer > 0
        else 0
    )

    packers_available_after_support = balancing["packers_available_after_support"]

    packers_assigned = min(
        packers_available_after_support,
        active_stations,
        needed_packers
    )

    reassigned_packers = max(
        0,
        packers_available_after_support - packers_assigned
    )

    line_capacity = packers_assigned * avg_units_per_packer

    units_packed = min(available_work, line_capacity)

    backlog_units = max(0, available_work - units_packed)

    utilization_rate = (
        units_packed / line_capacity
        if line_capacity > 0
        else 0
    )

    capacity_shortfall = max(
        0,
        available_work - line_capacity
    )

    bottleneck_flag = int(
        backlog_units > config["backlog_high"]
        or utilization_rate > 0.95 and backlog_units > 0
        or capacity_shortfall > 200
    )

    return {
        "active_stations": active_stations,
        "available_work": available_work,
        "needed_packers": needed_packers,
        "packers_assigned": packers_assigned,
        "reassigned_packers": reassigned_packers,
        "line_capacity": line_capacity,
        "units_packed": units_packed,
        "backlog_units": backlog_units,
        "utilization_rate": round(utilization_rate, 3),
        "capacity_shortfall": capacity_shortfall,
        "bottleneck_flag": bottleneck_flag
    }

# *Generate the dataset* #

In [ ]:
def generate_line_dataset(config=CONFIG):
    np.random.seed(config["seed"])

    timestamps = pd.date_range(                   # Create a row for each hour
        start=config["start_date"],
        periods=config["days"] * 24,
        freq="h"
    )

    lhs_df = create_lhs_samples(                  # Use Latin Hypercube for each hour
        n_rows=len(timestamps),
        seed=config["seed"]
    )

    rows = []
    previous_backlog = 0

    for i, timestamp in enumerate(timestamps):
        sample_row = lhs_df.iloc[i]

        inputs = generate_operational_inputs(     # Get planned work, available packers, etc.
            sample_row,
            timestamp,
            config
        )

        active_stations = max(0, config["total_stations"] - inputs["stations_down"])

        # Determine balancing needs based on backlog, planned work, available packers and stations, etc.
        balancing = apply_line_balancing(
            previous_backlog=previous_backlog,
            planned_work=inputs["planned_work"],
            packers_available=inputs["packers_available"],
            active_stations=active_stations,
            building_extra_packers=inputs["building_extra_packers"],
            other_line_capacity=inputs["other_line_capacity"],
            config=config
        )

        # Determine the line's throughput based on work, balancing, backlog, and available stations
        throughput = calculate_line_throughput(
            inputs=inputs,
            balancing=balancing,
            previous_backlog=previous_backlog,
            active_stations=active_stations,
            config=config
            )

        # Create the sample to add to the dataset
        row = {
            "timestamp": timestamp,
            "date": timestamp.date(),
            "hour": timestamp.hour,
            "line_id": config["line_id"],
            "total_stations": config["total_stations"],
            **inputs,
            **balancing,
            **throughput
        }

        rows.append(row)

        previous_backlog = throughput["backlog_units"]

    df = pd.DataFrame(rows)

    # Creates the "target" by asking "What will happen next hour?"
    df["next_hour_backlog"] = df["backlog_units"].shift(-1)

    df["next_hour_backlog_risk"] = (
        df["next_hour_backlog"] > config["backlog_high"]
    ).astype(int)

    df = df.dropna().reset_index(drop=True)

    return df

In [ ]:
df = generate_line_dataset(CONFIG)

df.to_csv("warehouse_line_throughput_synthetic.csv", index=False)

df.head()

,timestamp,date,hour,line_id,total_stations,shift,weekday,planned_work,stations_down,packers_scheduled,...,packers_assigned,reassigned_packers,line_capacity,units_packed,backlog_units,utilization_rate,capacity_shortfall,bottleneck_flag,next_hour_backlog,next_hour_backlog_risk
0,2026-04-01 00:00:00,2026-04-01,0,PACK-LINE-01,18,night,Wednesday,1071,2,10,...,10,0,820,820,251,1.0,251,1,584.0,0
1,2026-04-01 01:00:00,2026-04-01,1,PACK-LINE-01,18,night,Wednesday,1153,0,10,...,10,0,820,820,584,1.0,584,1,641.0,0
2,2026-04-01 02:00:00,2026-04-01,2,PACK-LINE-01,18,night,Wednesday,761,0,10,...,8,0,704,704,641,1.0,641,1,1129.0,1
3,2026-04-01 03:00:00,2026-04-01,3,PACK-LINE-01,18,night,Wednesday,1152,2,10,...,8,0,664,664,1129,1.0,1129,1,783.0,1
4,2026-04-01 04:00:00,2026-04-01,4,PACK-LINE-01,18,night,Wednesday,804,2,10,...,11,0,990,990,783,1.0,783,1,1051.0,1
